# COLUMNTRANSFORMERS + PIPELINES

In [31]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

In [32]:
df = pd.read_csv('train.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [33]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [34]:
df.shape

(891, 12)

SEPARATING COLUMNS USING SELECT_DTYPES METHOD

In [35]:
categorical_columns = df.select_dtypes(include = 'object').columns
numerical_columns = df.select_dtypes(exclude = 'object').columns
print(categorical_columns)
print(numerical_columns)

Index(['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked'], dtype='str')
Index(['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare'], dtype='str')


C:\Users\Pardhiv Reddy\AppData\Local\Temp\ipykernel_6472\2152062777.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include = 'object').columns


In [36]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [37]:
df.drop('Cabin',axis =1,inplace = True)

In [38]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Embarked         2
dtype: int64

WE DONT NEED SCALING BUT NEED IMPUTING AND ENCODING FOR SEX AND EMBARKED FEATURES
HERE WE NEED TRANSFORMATIONS FOR FEATURES [AGE,SEX,EMBARKED]

In [39]:
cat_col = ['Sex','Embarked']
num_col = ['Age','Fare']
num_transformer = Pipeline([
    ('imputer',SimpleImputer(strategy = 'median')),
    ('scaling',StandardScaler())
])
cat_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(sparse_output = False,handle_unknown="ignore"))
])
preprocessing = ColumnTransformer([
    ('num',num_transformer,num_col),
    ('cat',cat_transformer,cat_col)
])

In [40]:
X = df.drop('Survived',axis = 1)
y = df['Survived']

In [42]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2,random_state=42)
X_train_trans = preprocessing.fit_transform(X_train)
X_test_trans = preprocessing.transform(X_test)
print(X_train_trans.shape)

(712, 7)


AFTER ENCODING FEATURES INCREASE

In [43]:
features = preprocessing.get_feature_names_out()
print(features)

['num__Age' 'num__Fare' 'cat__Sex_female' 'cat__Sex_male'
 'cat__Embarked_C' 'cat__Embarked_Q' 'cat__Embarked_S']


CREATING A DATA FRAME OF PROCESSED FEATURES

In [44]:
X_train_df = pd.DataFrame(X_train_trans,columns = features)
X_train_df.head()

,num__Age,num__Fare,cat__Sex_female,cat__Sex_male,cat__Embarked_C,cat__Embarked_Q,cat__Embarked_S
0,1.253641,-0.078684,0.0,1.0,0.0,0.0,1.0
1,-0.477284,-0.377145,0.0,1.0,0.0,0.0,1.0
2,0.215086,-0.474867,0.0,1.0,0.0,0.0,1.0
3,-0.246494,-0.476230,0.0,1.0,0.0,0.0,1.0
4,-1.785093,-0.025249,1.0,0.0,0.0,0.0,1.0
